<a href="https://colab.research.google.com/github/eduhuemar001/llm-finetuning/blob/main/sentiment-data-classify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Libraries

In [15]:
!pip install transformers torch huggingface_hub
import re
import json
from tqdm import tqdm
from transformers import pipeline
from google.colab import files
from huggingface_hub import login

In [16]:
login()

### Preprocessing

In [17]:
uploaded = files.upload()

comments = []
with open("youtube_comments_clean.txt", "r", encoding="utf-8") as f:
    for line in f:
        text = line.strip()
        if text:
            comments.append(text)

comments = comments[:500]
print(f"Loaded {len(comments)} comments")
print("Sample:", comments[0])

Saving youtube_comments_clean.txt to youtube_comments_clean (2).txt
Loaded 500 comments
Sample: Ich liebe die neuen „Idee-Losen“ Spiele … Grüsse gehen raus an das PP-Team


### BERT pipeline




In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline

def load_pipeline(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda")
    return TextClassificationPipeline(model=model, tokenizer=tokenizer, device=0)

model1 = load_pipeline("oliverguhr/german-sentiment-bert")
model2 = load_pipeline("cardiffnlp/twitter-xlm-roberta-base-sentiment")
model3 = load_pipeline("nlptown/bert-base-multilingual-uncased-sentiment")

Device set to use cuda:0
Device set to use cuda:0


tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Device set to use cuda:0


In [21]:
def normalize(label):
    label = label.lower()
    if "neg" in label:
        return "negative"
    elif "pos" in label:
        return "positive"
    elif "neu" in label:
        return "neutral"
    return "neutral"

def majority_vote(text):
    preds = [
        normalize(model1(text)[0]["label"]),
        normalize(model2(text)[0]["label"]),
        normalize(model3(text)[0]["label"])
    ]
    vote = max(set(preds), key=preds.count)
    return vote if preds.count(vote) >= 2 else "uncertain"

output_file = "youtube_comments_labeled.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for text in tqdm(comments, desc="Labeling"):
        label = majority_vote(text)
        json.dump({"text": text, "label": label}, f, ensure_ascii=False)
        f.write("\n")

print(f"✅ Saved to {output_file}")

Labeling: 100%|██████████| 500/500 [00:17<00:00, 28.75it/s]

✅ Saved to youtube_comments_labeled.jsonl


In [22]:
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>